In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler, label_binarize

In [32]:
RANDOM_STATE = 42
TO_DROP = [
    'StartTime',
    'LastTime',
    # NOTE: src and dst cannot remove for graph probably
    'SrcAddr',
    'DstAddr',
    'sIpId',
    'dIpId',
    # drop target because we are predicting traffic
    'Target'
]

In [33]:
df = pd.read_csv("../../data/wustl_iiot_2021.csv")
df.head()

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,SAppBytes,DAppBytes,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target
0,2019-08-19 12:23:28,2019-08-19 12:23:28,192.168.0.20,192.168.0.2,0,59034,502,10,8,18,...,24,20,44,0.001176,0.053037,0,0.000000,0.0,normal,0
1,2019-08-19 15:13:24,2019-08-19 15:13:24,192.168.0.20,192.168.0.2,0,55841,502,10,8,18,...,24,20,44,0.001308,0.052681,0,0.000000,0.0,normal,0
2,2019-08-19 13:41:31,2019-08-19 13:41:31,192.168.0.20,192.168.0.2,0,63774,502,10,8,18,...,24,20,44,0.000690,0.051793,0,0.000000,0.0,normal,0
3,2019-08-19 12:43:19,2019-08-19 12:43:20,209.240.235.92,192.168.0.2,0,61771,80,4,0,4,...,0,0,0,0.000000,0.889555,0,419.338813,0.0,DoS,1
4,2019-08-19 14:49:44,2019-08-19 14:49:48,192.168.0.20,192.168.0.1,3,0,0,14,0,14,...,476,0,476,0.000000,3.500055,0,525.146562,0.0,normal,0


In [34]:
df["StartTime"] = pd.to_datetime(df["StartTime"])
df["LastTime"] = pd.to_datetime(df["LastTime"])

negative_duration_mask = df['StartTime'] > df['LastTime']
df['StartTime'], df['LastTime'] = (
    np.where(negative_duration_mask, df['LastTime'], df['StartTime']),
    np.where(negative_duration_mask, df['StartTime'], df['LastTime'])
)

df = df.sort_values("StartTime").reset_index(drop=True)
df["nearest_sec"] = df["StartTime"].dt.floor("s")

In [35]:
proto_dummies = pd.get_dummies(
    df["Proto"],
    prefix="proto",
    dtype=int
)

df = pd.concat([df, proto_dummies], axis=1)

In [36]:
stos_dummies = pd.get_dummies(
    df['sTos'],
    prefix='stos',
    dtype=int
)

df = pd.concat([df, stos_dummies], axis=1)

In [37]:
df.head()

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,proto_6,proto_17,proto_58,proto_2054,proto_35020,stos_0,stos_16,stos_126,stos_192,stos_207
0,2019-08-19 09:45:18,2019-08-19 09:46:18,0,5093,0,0,2,52380,2845,52380,...,0,0,0,0,0,1,0,0,0,0
1,2019-08-19 09:46:03,2019-08-19 09:46:03,192.168.0.20,192.168.0.2,0,60840,502,10,8,18,...,1,0,0,0,0,1,0,0,0,0
2,2019-08-19 09:46:03,2019-08-19 09:46:03,192.168.0.20,192.168.0.2,0,60849,502,10,8,18,...,1,0,0,0,0,1,0,0,0,0
3,2019-08-19 09:46:03,2019-08-19 09:46:03,192.168.0.20,192.168.0.2,0,60845,502,10,8,18,...,1,0,0,0,0,1,0,0,0,0
4,2019-08-19 09:46:03,2019-08-19 09:46:03,192.168.0.20,192.168.0.2,0,60863,502,10,8,18,...,1,0,0,0,0,1,0,0,0,0


In [38]:
df["Dport"].value_counts()

Dport
502      1103533
80         52935
0          11114
8080        7221
1740        4812
          ...   
33773          1
17841          1
62222          1
63904          1
49343          1
Name: count, Length: 7781, dtype: int64

In [39]:
def port_range(port):
    if port <= 1023:
        return "well_known"
    elif port <= 49151:
        return "registered"
    else:
        return "dynamic"

# NOTE: the dataset does not contain all the ports checked by the function below
# so the aggregation only takes those columns that are created by the one hot encoding
def dst_port_service(port):
    if port in [20, 21]:
        return "ftp"
    elif port == 22:
        return "ssh"
    elif port == 23:
        return "telnet"
    elif port == 53:
        return "dns"
    elif port in [80, 8080]:
        return "http"
    elif port == 443:
        return "https"
    elif port == 445:
        return "smb"
    elif port == 3389:
        return "rdp"
    elif port == 502:
        return "modbus"
    elif port <= 1023:
        return "other_well_known"
    elif port <= 49151:
        return "registered"
    else:
        return "dynamic"

src_port_features = pd.get_dummies(
    df['Sport'].apply(port_range),
    prefix='src_port',
    dtype=int
)

dst_port_features = pd.get_dummies(
    df["Dport"].apply(dst_port_service),
    prefix="dport",
    dtype=int
)

df = pd.concat([df, src_port_features], axis=1)
df = pd.concat([df, dst_port_features], axis=1)

In [ ]:
per_second = (
    df.groupby(["SrcAddr", "DstAddr", "nearest_sec"])
      .agg(
        mean=("Mean", "sum"),

        # instead of aggregating all the unique source and dest ports, take the number of unique ports used instead
        num_src_ports=('Sport', "nunique"),
        num_dst_ports=('Dport', "nunique"),
        src_port_dynamic=("src_port_dynamic", "max"),
        src_port_registered=("src_port_registered", "max"),
        src_port_well_known=("src_port_well_known", "max"),
        dport_dynamic=("dport_dynamic", "max"),
        dport_ftp=("dport_ftp", "max"),
        dport_http=("dport_http", "max"),
        dport_https=("dport_https", "max"),
        dport_modbus=("dport_modbus", "max"),
        dport_other_well_known=("dport_other_well_known", "max"),
        dport_registered=("dport_registered", "max"),

        src_pkts=("SrcPkts", "sum"),
        dst_pkts=("DstPkts", "sum"),
        tot_pkts=("TotPkts", "sum"),

        src_bytes=("SrcBytes", "sum"),
        dst_bytes=("DstBytes", "sum"),
        tot_bytes=("TotBytes", "sum"),

        src_load=("SrcLoad", "sum"),
        dst_load=("DstLoad", "sum"),
        tot_load=("Load", "sum"),

        src_rate=("SrcRate", "sum"),
        dst_rate=("DstRate", "sum"),
        tot_rate=("Rate", "sum"),

        src_loss=("SrcLoss", "sum"),
        dst_loss=("DstLoss", "sum"),
        loss=("Loss", "sum"),

        # recompute pLoss later

        proto_0=('proto_0', 'max'),
        proto_1=('proto_1', 'max'),
        proto_2=('proto_2', 'max'),
        proto_6=('proto_6', 'max'),
        proto_17=('proto_17', 'max'),
        proto_58=('proto_58', 'max'),
        proto_2054=('proto_2054', 'max'),
        proto_35020=('proto_35020', 'max'),
        
        duration_mean=("Dur", "mean"),
        duration_min=("Dur", "min"),
        duration_max=("Dur", "max"),
        duration_sum=("Dur", "sum"),

        tcp_rtt_mean=('TcpRtt', 'mean'),

        idle_mean=('IdleTime', 'mean'),

        src_dsb_mean=("sDSb", "mean"),
        src_ttl_mean=("sTtl", "mean"),
        dst_ttl_mean=("dTtl", "mean"),

        src_app_bytes=("SAppBytes", "sum"),
        dst_app_bytes=("DAppBytes", "sum"),
        tot_app_bytes=("TotAppByte", "sum"),

        syn_ack_mean=("SynAck", "mean"),

        run_time_sum=("RunTime", "sum"),

        stos_0=("stos_0", "max"),
        stos_16=("stos_16", "max"),
        stos_126=("stos_126", "max"),
        stos_192=("stos_192", "max"),
        stos_207=("stos_207", "max"),

        src_jitter_mean=("SrcJitter", "mean"),
        dst_jitter_mean=("DstJitter", "mean"),

        src_int_pkt_mean=("SIntPkt", "mean"),
        dst_int_pkt_mean=("DIntPkt", "mean"),
        

        # src_rate_mean=("SrcRate", "mean"),
        # dst_rate_mean=("DstRate", "mean"),
        # rate_mean=("Rate", "mean"),

        # src_load_mean=("SrcLoad", "mean"),
        # dst_load_mean=("DstLoad", "mean"),

        # src_jitter_mean=("SrcJitter", "mean"),
        # dst_jitter_mean=("DstJitter", "mean"),

        # tcp_rtt_mean=("TcpRtt", "mean"),
        traffic=("Traffic", "first")
      )
      .reset_index()
)

In [41]:
per_second

,SrcAddr,DstAddr,nearest_sec,mean,num_src_ports,num_dst_ports,src_port_dynamic,src_port_registered,src_port_well_known,dport_dynamic,...,stos_0,stos_16,stos_126,stos_192,stos_207,src_jitter_mean,dst_jitter_mean,src_int_pkt_mean,dst_int_pkt_mean,traffic
0,0,0,2019-08-19 09:46:08,0,1,1,0,0,1,0,...,1,0,0,0,0,0.000000,0.0,0.000000,0.0,normal
1,0,0,2019-08-19 13:24:34,0,1,1,0,0,1,0,...,1,0,0,0,0,0.000000,0.0,0.000000,0.0,normal
2,0,14740,2019-08-19 11:04:18,0,1,1,0,0,1,0,...,1,0,0,0,0,0.000000,0.0,0.000000,0.0,normal
3,0,15046,2019-08-19 12:29:18,0,1,1,0,0,1,0,...,1,0,0,0,0,0.000000,0.0,0.000000,0.0,normal
4,0,16274,2019-08-19 11:09:18,0,1,1,0,0,1,0,...,1,0,0,0,0,0.000000,0.0,0.000000,0.0,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46473,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 15:39:16,0,1,1,1,0,0,0,...,1,0,0,0,0,47.176043,0.0,33.377000,0.0,normal
46474,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 15:54:16,0,1,1,1,0,0,0,...,1,0,0,0,0,46.890844,0.0,33.172332,0.0,normal
46475,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 16:09:15,0,1,1,1,0,0,0,...,1,0,0,0,0,47.048766,0.0,33.280000,0.0,normal
46476,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 16:24:15,0,1,1,1,0,0,0,...,1,0,0,0,0,47.100148,0.0,33.314332,0.0,normal


In [ ]:
numeric_features = [
    'mean',
    'num_src_ports',
    'num_dst_ports',
    'src_pkts',
    'dst_pkts',
    'tot_pkts',
    'src_bytes',
    'dst_bytes',
    'tot_bytes',
    'src_load',
    'dst_load',
    'tot_load',
    'src_rate',
    'dst_rate',
    'tot_rate',
    'src_loss',
    'dst_loss',
    'loss',
    'duration_mean',
    'duration_min',
    'duration_max',
    'duration_sum',
    'tcp_rtt_mean',
    'idle_mean',
    'src_dsb_mean',
    'src_ttl_mean',
    'dst_ttl_mean',
    'src_app_bytes',
    'dst_app_bytes',
    'tot_app_bytes',
    'syn_ack_mean',
    'run_time_sum',
    'src_jitter_mean',
    'dst_jitter_mean',
    'src_int_pkt_mean',
    'dst_int_pkt_mean'
]

categorical_features = [
    'proto_0', 'proto_1', 'proto_2', 'proto_6',
    'proto_17', 'proto_58', 'proto_2054', 'proto_35020',
    'stos_0', 'stos_16', 'stos_126', 'stos_192', 'stos_207',
    "src_port_dynamic", "src_port_registered", "src_port_well_known",
    "dport_dynamic", "dport_ftp", "dport_http", "dport_https","dport_modbus",
    "dport_other_well_known", "dport_registered"
]

feature_cols = numeric_features.copy() + categorical_features.copy()

target_col = 'traffic'

In [43]:
def resample_pair(group):

    group = group.sort_values("nearest_sec")
    group = group.set_index("nearest_sec")

    full_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="1s"
    )

    group = group.reindex(full_index)
    group.index.name = "nearest_sec"
    return group

In [44]:
PRIORITY = ['Backdoor', 'CommInj', 'Reconn', 'DoS', 'normal']

def extract_y(y_seq: np.ndarray):
    # case 1: all nan
    if pd.isna(y_seq).all():
        return "normal"

    labels = y_seq[~pd.isna(y_seq)]
    unique_labels = np.unique(labels)

    # return highest PRIORITY label present, lower index means higher PRIORITY
    for label in PRIORITY:
        if label in unique_labels:
            return label

    return "normal"

In [45]:
SEQUENCE_LENGTH = 5

X = []
y = []
pairs = []

for (src, dst), group in per_second.groupby(["SrcAddr", "DstAddr"]):
    group = group.sort_values("nearest_sec")
    resampled_group = resample_pair(group)

    features = resampled_group[feature_cols]
    labels = resampled_group["traffic"].values

    # fill null values for features here
    # since the categorical features are one-hot encoded, all 0 for that feature denotes not present
    features[numeric_features] = features[numeric_features].fillna(0)
    features[categorical_features] = features[categorical_features].fillna(0)
    features = features.values

    # pad with zeros if the group is smaller than sequence length
    if len(resampled_group) < SEQUENCE_LENGTH:
        pad_length = SEQUENCE_LENGTH - len(resampled_group)

        features = np.pad(
            features,
            pad_width=((0, pad_length), (0, 0)),
            mode="constant",
            constant_values=0
        )

    for i in range(0, len(resampled_group) - SEQUENCE_LENGTH + 1, SEQUENCE_LENGTH):
        X_seq = features[i:i + SEQUENCE_LENGTH]
        y_seq = labels[i: i + SEQUENCE_LENGTH]
        y_val = extract_y(y_seq)

        X.append(X_seq)
        y.append(y_val)
        pairs.append((src, dst))

X = np.array(X)
y = np.array(y)
pairs = np.array(pairs)

# 70/10/20 split
X_train, X_test, y_train, y_test, pairs_train, pairs_test = train_test_split(
    X, y, pairs, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val, pairs_train, pairs_val = train_test_split(
    X_train, y_train, pairs_train, test_size=0.125,
    random_state=RANDOM_STATE, stratify=y_train
)

In [46]:
# NOTE: since feature_cols = numeric_features + categorical_features, we can rely on the indexing to be the same
n_numeric = len(numeric_features)

def scale_numeric(X, scaler, fit=False):
    X_numeric = X[:, :, :n_numeric].reshape(-1, n_numeric)
    X_numeric = scaler.fit_transform(X_numeric) if fit else scaler.transform(X_numeric)
    X[:, :, :n_numeric] = X_numeric.reshape(X.shape[0], X.shape[1], n_numeric)
    return X
    
scaler = StandardScaler()
X_train = scale_numeric(X_train, scaler, fit=True)
X_val = scale_numeric(X_val, scaler)
X_test = scale_numeric(X_test, scaler)

In [47]:
np.unique(y_train)

array(['Backdoor', 'CommInj', 'DoS', 'Reconn', 'normal'], dtype='<U8')

In [48]:
# index 0 represents backdoor, 1 represents comminj, and so on
CLASSES = ['Backdoor', 'CommInj', 'DoS', 'Reconn', 'normal']

y_train = label_binarize(y_train, classes=CLASSES)
y_val = label_binarize(y_val, classes=CLASSES)
y_test = label_binarize(y_test, classes=CLASSES)

In [49]:
np.save('../../data/processed/X_train.npy', X_train)
np.save('../../data/processed/X_val.npy', X_val)
np.save('../../data/processed/X_test.npy', X_test)

np.save('../../data/processed/pairs_train.npy', pairs_train)
np.save('../../data/processed/pairs_val.npy', pairs_val)
np.save('../../data/processed/pairs_test.npy', pairs_test)

np.save('../../data/processed/y_train.npy', y_train)
np.save('../../data/processed/y_val.npy', y_val)
np.save('../../data/processed/y_test.npy', y_test)